In [0]:
%sql
SELECT _source_year, _source_month, _ingested_at, COUNT(*) AS rows
FROM nyc_taxi.bronze.yellow_tripdata
WHERE _source_year = 2024 AND _source_month = 1
GROUP BY _source_year, _source_month, _ingested_at
ORDER BY _ingested_at;

In [0]:
%sql
DELETE FROM nyc_taxi.bronze.yellow_tripdata
WHERE _source_year = 2024
  AND _source_month = 1
  AND _ingested_at = (
    SELECT MAX(_ingested_at)
    FROM nyc_taxi.bronze.yellow_tripdata
    WHERE _source_year = 2024 AND _source_month = 1
  );

In [0]:
print("bronze total:", spark.table("nyc_taxi.bronze.yellow_tripdata").count())
# expect exactly 41,169,720 — matching Week 1's original confirmed count

In [0]:
path = "/Workspace/Users/rmdd.abreu@gmail.com/nyc-taxi-fabric-lakehouse/decisions.md"
entry = """
2026-07-28 — Testing 01_bronze_ingest standalone re-triggered ingest_month(2024, 1) via the widget default values, duplicating January 2024 in Bronze (Bronze has no dedup by design — that's Silver's job). Caught via _ingested_at grouping (two distinct batches for the same month), fixed with a targeted DELETE on the newer batch rather than dropping the whole table. Confirms the exact risk the notebook's own warning cell was written to flag.
"""
with open(path, "a") as f:
    f.write(entry)
print("done")